# Counterfactual feasibility results
Run `python3 run_pipeline.py` first. DiCE Level 1/2 are diagnostic only; only model-valid and HR-feasible Level 3 scenarios can be selected. Here, *HR-feasible* means feasible under the encoded study rules, not approved for a real employee decision.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tables = ROOT / 'outputs' / 'tables'
scenarios = ROOT / 'outputs' / 'scenarios'
assessment = pd.read_csv(tables / 'hr_intervention_feasibility_assessment.csv')
level1 = pd.read_csv(scenarios / 'dice_level_1.csv')
level2 = pd.read_csv(scenarios / 'dice_level_2.csv')
level3 = pd.read_csv(scenarios / 'level_3_grid.csv')
display(assessment.groupby(['analysis_status', 'final_group'], dropna=False).size().rename('employees').reset_index())
display(pd.read_csv(tables / 'counterfactual_found_rate_by_level.csv'))

## Employee-level demo
Set `employee_id` to any ID in the assessment table. The demo shows the final classification and the model-valid scenarios found at each level. Level 1/2 results remain diagnostic even when they cross the score threshold.


In [ ]:
def show_employee(employee_id: int) -> None:
    employee = assessment.loc[assessment['employee_id'].eq(employee_id)]
    if employee.empty:
        valid_ids = assessment['employee_id'].astype(int).tolist()
        raise ValueError(f'Employee {employee_id} is not in the high-risk cohort. Choose one of: {valid_ids}')

    summary_fields = [
        'employee_id', 'attrition_score', 'high_risk_threshold',
        'model_valid_scenario_found', 'level3_feasible_scenario_found',
        'best_scenario_id', 'best_changed_features', 'score_after_best_scenario',
        'salary_increase_pct', 'stock_option_increment', 'overtime_change',
        'violation_reason_if_none', 'final_group', 'analysis_status',
    ]
    display(employee[summary_fields].T.rename(columns={employee.index[0]: 'value'}))

    scenario_fields = [
        'selected_best', 'constraint_level', 'scenario_id', 'score_before', 'score_after',
        'changed_features', 'changes_json', 'n_changes', 'salary_increase_pct',
        'stock_option_increment', 'overtime_change', 'hr_feasible', 'violations',
    ]
    best_id = employee.iloc[0]['best_scenario_id']
    for level, frame in ((1, level1), (2, level2), (3, level3)):
        rows = frame.loc[frame['employee_id'].eq(employee_id) & frame['model_valid'].astype(bool)].copy()
        rows['selected_best'] = rows['scenario_id'].eq(best_id)
        rows = rows.sort_values(['selected_best', 'hr_feasible', 'score_after'], ascending=[False, False, True])
        print(f'Level {level}: {len(rows)} model-valid scenario(s)')
        display(rows[scenario_fields].head(10) if not rows.empty else pd.DataFrame(columns=scenario_fields))

employee_id = 144  # Try 269 for a model-valid but not HR-feasible case.
show_employee(employee_id)

In [ ]:
# Robustness checks: these are aggregate diagnostics, not employee-level causal effects.
display(pd.read_csv(tables / 'sensitivity_threshold.csv'))
display(pd.read_csv(tables / 'sensitivity_salary_cap.csv'))
display(pd.read_csv(tables / 'sensitivity_overtime.csv'))
display(pd.read_csv(tables / 'stability_by_seed.csv'))